In [ ]:
# Name of current environment
dbutils.widgets.dropdown(
    "env",
    "local",
    ["local", "dev", "staging", "prod"],
    "Environment Name"
)

env = dbutils.widgets.get("env")

if env == "local":
    db_env = "dev"
else:
    db_env = env

dbutils.widgets.dropdown(
    "run_historical_operational_monitoring_job",
    "no",
    ["no", "yes"], # no
    "run historical operational monitoring job?"
)

dbutils.widgets.text("start_date", "", "Start Date ('YYYY-MM-DD')")
dbutils.widgets.text("end_date", "", "End Date ('YYYY-MM-DD')")

In [ ]:
run_historical_operational_monitoring_job = dbutils.widgets.get("run_historical_operational_monitoring_job")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
is_run_historical_operational_monitoring_job = run_historical_operational_monitoring_job.strip().lower() == 'yes'

print("is_run_historical_operational_monitoring_job", is_run_historical_operational_monitoring_job)
print("start_date", start_date)
print("end_date", end_date)

In [ ]:
import os
import yaml
import sys

notebook_path = "/Workspace/" + os.path.dirname(
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

# Define the paths to your configuration files
config_common_path = os.path.join(notebook_path, f"../../config/config-common-decline-letter.yml")
config_common_env = os.path.join(notebook_path, f"../../config/{env}/config-decline-letter.yml")

# Load configuration
with open(config_common_path, "rt") as f:
    config = yaml.safe_load(f)
with open(config_common_env, "r") as f:
    config.update(yaml.safe_load(f))

catalog_name = config['catalog_gold']
schema_name = config['schema_name']
app_space_id = config['app_space']

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when
from pyspark.sql.window import Window as W

In [ ]:
# Table names
agent_inference = f"{catalog_name}.{schema_name}.{config['monitoring_config']['data_sources']['source_table']}"
agent_metrics = f"{catalog_name}.{schema_name}.{config['monitoring_config']['data_sources']['target_table']}"

# Token cost
cost_per_1k_input_tokens = config['monitoring_config']['cost_settings']['cost_per_1k_input_tokens']
cost_per_1k_output_tokens = config['monitoring_config']['cost_settings']['cost_per_1k_output_tokens']
exchange_rate_usd_to_aud = config['monitoring_config']['cost_settings']['exchange_rate_usd_to_aud']

print(f'app_space_id: {app_space_id}')
print(f'catalog_name: {catalog_name}')
print(f'schema_name: {schema_name}')
print(f'agent_inference_table: {agent_inference}')
print(f'agent_metrics_table: {agent_metrics}')

In [ ]:
def read_and_extract_table(
    source_table: str,
    target_table: str,
    is_run_historical_operational_monitoring_job: bool,
    start_date: str = None, # 'YYYY-MM-DD'
    end_date: str = None    # 'YYYY-MM-DD'
):
    if is_run_historical_operational_monitoring_job:
        assert start_date is not None and end_date is not None, \
            "For historical runs, start_date and end_date (YYYY-MM-DD) must be provided."
        
        filter_query = f"""SELECT 
            databricks_request_id,
            request_date_time_aedt,
            request_date_aedt,
            execution_duration_ms,
            user_id AS session_id,
            total_input_tokens as prompt_tokens,
            total_output_tokens as completion_tokens,
            (total_input_tokens + total_output_tokens) AS total_tokens,
            user_approved,
            status_code,
            ai_letter,
            user_fb
        FROM {source_table} s
        WHERE s.request_date_aedt BETWEEN DATE('{start_date}') AND DATE('{end_date}')"""
    else:
        filter_query = f"""
        -- fetch request_date_aedt and start from there for incremental load
        WITH tgt_max AS (
            SELECT MAX(request_date_aedt) AS last_date_aedt FROM {target_table}
        ),
        defaults AS (
            SELECT COALESCE((SELECT last_date_aedt FROM tgt_max), DATE('1970-01-01')) AS last_date_aedt
        ),
        src AS (
            SELECT 
                databricks_request_id,
                request_date_aedt,
                request_date_time_aedt,
                execution_duration_ms,
                user_id AS session_id,
                total_input_tokens,
                total_output_tokens,
                user_approved,
                status_code,
                ai_letter,
                user_fb
            FROM {source_table} s
        ),
        eligible AS (
            SELECT e.*
            FROM src e
            CROSS JOIN defaults d
            WHERE e.request_date_aedt >= d.last_date_aedt
        )
        SELECT 
            databricks_request_id,
            request_date_aedt,
            request_date_time_aedt,
            execution_duration_ms,
            session_id,
            total_input_tokens as prompt_tokens,
            total_output_tokens as completion_tokens,
            (total_input_tokens + total_output_tokens) AS total_tokens,
            user_approved,
            status_code,
            ai_letter,
            user_fb
        FROM eligible
        """
    
    print(f"query {filter_query}")
    new_data_df = spark.sql(filter_query)
    return new_data_df

In [ ]:
def calculate_token_cost(filtered_df, cost_per_1k_input_tokens: float, cost_per_1k_output_tokens: float, exchange_rate_usd_to_aud: float):
    # Filter for records where endpoint is 'payg' and calculate costs
    df_with_costs = filtered_df.withColumn(
        "cost_input_usd",
        when(col("endpoint_type") == "payg", (col("prompt_tokens") / 1000) * cost_per_1k_input_tokens)
    ).withColumn(
        "cost_output_usd",
        when(col("endpoint_type") == "payg", (col("completion_tokens") / 1000) * cost_per_1k_output_tokens)
    ).withColumn(
        "cost_input_aud",
        when(col("endpoint_type") == "payg", col("cost_input_usd") * exchange_rate_usd_to_aud)
    ).withColumn(
        "cost_output_aud",
        when(col("endpoint_type") == "payg", col("cost_output_usd") * exchange_rate_usd_to_aud)
    ).withColumn(
        "total_cost_aud",
        when(col("endpoint_type") == "payg", col("cost_input_aud") + col("cost_output_aud"))
    )
    return df_with_costs

# Execution
filtered_df = read_and_extract_table(agent_inference, agent_metrics, is_run_historical_operational_monitoring_job, start_date, end_date)
filtered_df = filtered_df\
    .withColumn("endpoint_type", F.lit("payg"))\
    .withColumn("model_used", F.lit(config["llm_config"]["deployment_name"]))

filtered_df = calculate_token_cost(filtered_df, cost_per_1k_input_tokens, cost_per_1k_output_tokens, exchange_rate_usd_to_aud)

In [ ]:
word_count = (
    F.when(F.col("ai_letter").isNull(), F.lit(0))
    .when(F.length(F.trim(F.col("ai_letter"))) == 0, F.lit(0))
    .otherwise(F.size(F.array_remove(F.split(
        F.regexp_replace(F.trim(F.col("ai_letter")), r"\s+", " "), " "), ""))).alias("word_count")
)

w = W.partitionBy("request_date_aedt", "session_id").orderBy(F.col("request_date_time_aedt").desc())

ai_letter_len_per_session = (
    filtered_df
    .withColumn("final_letter_len", F.row_number().over(w))
    .filter(F.col("final_letter_len") == 1)
    .select("request_date_aedt",
            "session_id",
            word_count.alias("word_count")
    )
)

ai_letter_len_df = ai_letter_len_per_session.groupby("request_date_aedt").agg(
    (F.sum("word_count") / F.count("session_id")).alias("avg_ai_letter_len")
)

ai_letter_len_df.show()

In [ ]:
# Step 1: Aggregate max duration per session
letter_gen_df = filtered_df.groupBy("request_date_aedt", "session_id").agg(
    F.max("execution_duration_ms").alias("letter_gen_time")
)

# Step 2: Aggregate average time per day (converting ms to minutes)
letter_gen_df = letter_gen_df.groupBy("request_date_aedt").agg(
    (F.sum("letter_gen_time") / (1000 * 60) / F.countDistinct("session_id")).alias("avg_letter_gen_time")
)

letter_gen_df.show()

In [ ]:
# Aggregate daily metrics
filtered_df = filtered_df.groupBy("request_date_aedt").agg(
    F.countDistinct("session_id").alias("num_of_letters_gen"),
    (F.count("session_id") / F.countDistinct("session_id")).alias("avg_num_conv"),
    ((F.sum("execution_duration_ms") / (1000 * 60)) / F.countDistinct("session_id")).alias("avg_session_time"),
    (F.sum("prompt_tokens") / F.countDistinct("session_id")).alias("avg_input_tokens"),
    (F.sum("completion_tokens") / F.countDistinct("session_id")).alias("avg_output_tokens"),
    (F.sum("total_cost_aud")).alias("total_token_cost"),
    ((F.sum("total_cost_aud") / F.countDistinct("session_id"))).alias("avg_token_cost"),
    (F.sum(F.col("user_approved").cast("int")) / F.countDistinct("session_id")).alias("approval_rate"),
    F.sum(F.when(F.col("status_code") != 200, 1).otherwise(0)).alias("error_count")
)

# Join the dataframes together
final_df = filtered_df.join(letter_gen_df, on='request_date_aedt', how='left')
final_df = final_df.join(ai_letter_len_df, on='request_date_aedt', how='left')

In [ ]:
from datetime import datetime, timedelta
from pyspark.sql import SparkSession

def write_to_target_table(df, target_table: str):
    """
    Upsert calculated metrics to target table.
    - If request_date_aedt exists: update metrics for that date
    - If request_date_aedt doesn't exist: insert new row
    """
    spark = SparkSession.builder.getOrCreate()

    # Add audit timestamp
    update_ts = datetime.utcnow() + timedelta(hours=10)  # AEDT
    df = df.withColumn("last_updated_timestamp", F.lit(update_ts))

    target_columns = df.columns

    # Check if target table exists
    try:
        spark.sql(f"DESCRIBE TABLE {target_table}")
        table_exists = True
    except Exception:
        table_exists = False

    if not table_exists:
        df.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"Created new table {target_table} with {df.count()} records")
    else:
        df.createOrReplaceTempView("new_metrics")

        update_cols = [c for c in target_columns if c != 'request_date_aedt']
        update_clause = ", ".join([f"target.{c} = new.{c}" for c in update_cols])
        insert_cols = ", ".join(target_columns)
        insert_vals = ", ".join([f"new.{c}" for c in target_columns])

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING new_metrics AS new
            ON target.request_date_aedt = new.request_date_aedt
            WHEN MATCHED THEN UPDATE SET {update_clause}
            WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
        """
        spark.sql(merge_sql)
        print(f"Merged {df.count()} records into {target_table}")

# Execute
write_to_target_table(final_df, agent_metrics)